# OOD Evaluation: Vercellotti (Context Modes)

This notebook runs OOD annotation on the Vercellotti corpus with four context modes:
- `utterance_only`
- `local_prev`
- `full_prev`
- `full_document`

Then it compares modes and exports a manual review sheet for changed outputs.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/shamira-venturini/talkbank-morphosyntax-error-annotator.git"
REPO_NAME = "talkbank-morphosyntax-error-annotator"
COLAB_REPO_ROOT = Path("/content") / REPO_NAME

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "scripts").exists() and (cur / "data").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise RuntimeError("Could not locate repository root from current working directory")

if Path("/content").exists():
    if not COLAB_REPO_ROOT.exists():
        clone_cmd = ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_ROOT)]
        print("Cloning repository:", " ".join(clone_cmd))
        subprocess.run(clone_cmd, check=True)
    repo_root = COLAB_REPO_ROOT
else:
    repo_root = find_repo_root(Path.cwd())

os.chdir(repo_root)
print("Repo root:", repo_root)
print("Python:", sys.executable)

In [ ]:
# Install runtime dependencies in Colab.
!pip -q install -U transformers peft bitsandbytes accelerate sentencepiece

In [ ]:
# Load HF token from Colab secrets when available.
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "HF_TOKEN is not set. Add it to Colab Secrets or os.environ."

In [ ]:
# Config
CORPUS_DIR = "data/OOD_data/Vercellotti"
PREPARED_DIR = "data/processed/ood_vercellotti"
RESULTS_DIR = "results/ood_vercellotti"

SPEAKER_POLICY = "dominant"      # dominant | first_participant | all
CONTEXT_SCOPE = "same_speaker"   # same_speaker | file_selected
LOCAL_PREV_K = 2

BATCH_SIZE = 8
FULL_DOCUMENT_BATCH_SIZE = 4
MAX_NEW_TOKENS = 96
MAX_SEQ_LENGTH = 384
MAX_CONTEXT_CHARS = 4000
LIMIT = 0

BASE_MODEL = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
ADAPTER_REPO = "mash-mash/talkbank-morphosyntax-annotator-final-recon_full_comp_preserve_final_seed3407"
CHAT_TOKENS = "experiments/recon_full_comp_preserve/chat_tokens.json"
STAGE3_SPLIT = "experiments/recon_full_comp_preserve/stage3_train.jsonl"

In [ ]:
# 1) Prepare utterance-level OOD dataset from .cha files
prep_cmd = [
    "python3", "scripts/prepare_ood_vercellotti_dataset.py",
    "--corpus-dir", CORPUS_DIR,
    "--out-dir", PREPARED_DIR,
    "--speaker-policy", SPEAKER_POLICY,
    "--min-word-count", "1",
]
print(" ".join(prep_cmd))
subprocess.run(prep_cmd, check=True)

prepared_jsonl = Path(PREPARED_DIR) / "vercellotti_utterances.jsonl"
assert prepared_jsonl.exists(), f"Missing prepared file: {prepared_jsonl}"

In [ ]:
# 2) Run OOD inference across context modes
modes = [
    ("utterance_only", BATCH_SIZE),
    ("local_prev", BATCH_SIZE),
    ("full_prev", BATCH_SIZE),
    ("full_document", FULL_DOCUMENT_BATCH_SIZE),
]

for mode, bs in modes:
    cmd = [
        "python3", "scripts/run_ood_context_inference.py",
        "--input-jsonl", str(prepared_jsonl),
        "--out-dir", RESULTS_DIR,
        "--context-mode", mode,
        "--local-prev-k", str(LOCAL_PREV_K),
        "--context-scope", CONTEXT_SCOPE,
        "--batch-size", str(bs),
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--max-seq-length", str(MAX_SEQ_LENGTH),
        "--max-context-chars", str(MAX_CONTEXT_CHARS),
        "--base-model", BASE_MODEL,
        "--adapter-repo", ADAPTER_REPO,
        "--chat-tokens", CHAT_TOKENS,
        "--stage3-split", STAGE3_SPLIT,
    ]
    if LIMIT > 0:
        cmd.extend(["--limit", str(LIMIT)])
    print("\n", " ".join(cmd))
    subprocess.run(cmd, check=True)

In [ ]:
# 3) Compare context modes and export review set
analysis_out = Path(RESULTS_DIR) / "context_analysis"
analysis_cmd = [
    "python3", "scripts/analyze_ood_context_modes.py",
    "--predictions-dir", RESULTS_DIR,
    "--baseline-mode", "utterance_only",
    "--out-dir", str(analysis_out),
]
print(" ".join(analysis_cmd))
subprocess.run(analysis_cmd, check=True)
print("Done:", analysis_out)

In [ ]:
# 4) Quick preview of key outputs
import pandas as pd

pairwise = pd.read_csv(analysis_out / "pairwise_vs_baseline.csv")
mode_summary = pd.read_csv(analysis_out / "context_mode_summary.csv")
print("Pairwise vs baseline")
display(pairwise)
print("\nMode summary")
display(mode_summary)

review_path = analysis_out / "manual_review_changed_outputs.csv"
print("\nManual review sheet:", review_path)
print("Rows:", pd.read_csv(review_path).shape[0])

In [ ]:
# Optional: zip results for local download
import shutil
from google.colab import files

zip_path = shutil.make_archive(str(Path(RESULTS_DIR)), "zip", root_dir=str(Path(RESULTS_DIR).parent), base_dir=Path(RESULTS_DIR).name)
print("Created:", zip_path)
files.download(zip_path)